# Semantic Search with Quora Questions

This notebook walks through a small local semantic-search system:

1. load a public Quora question dataset
2. create embeddings with a sentence-transformer model
3. store the vectors in a local Chroma collection
4. query for semantically similar questions

Run the notebook top to bottom the first time. The initial embedding pass can take a few minutes depending on your machine.

In [1]:
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import chromadb
import numpy as np
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

In [ ]:
RAW_LIMIT = 2000
DATASET_NAME = "sentence-transformers/quora-duplicates"
DATASET_CONFIG = "pair-class"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "quora_semantic_search"
DB_PATH = Path("../chroma_db")

split_name = f"train[:{RAW_LIMIT}]"
dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split=split_name)
df = dataset.to_pandas()
questions = (
    pd.concat([df["sentence1"], df["sentence2"]])
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Loaded {len(df):,} rows and {len(questions):,} unique questions")
questions.head()

Loaded 2,000 rows and 3,978 unique questions


0    What is the step by step guide to invest in sh...
1    What is the story of Kohinoor (Koh-i-Noor) Dia...
2    How can I increase the speed of my internet co...
3    Why am I mentally very lonely? How can I solve...
4    Which one dissolve in water quikly sugar, salt...
dtype: object

In [ ]:
questions_list = questions.tolist()
questions.sample(10, random_state=42).tolist()

['Do you regret your divorce?',
 'Who were the creators of trigonometry?',
 'How much is the maximum gold you can take when flying from India to USA?',
 'What are the top 5 things I should definitely not do when I am on a flight?',
 'How can I write to Narendra Modi?',
 'What websites do you use everyday?',
 'How much did Microsoft pay for LiveLoop?',
 'Is a better translation of "exceptio probat regulum" "the exception tests the rule?"',
 'How can I get entry in MIT?',
 'Are there any good horror movies in 2016?']

In [17]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    questions_list,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

embeddings.shape

Batches: 100%|██████████| 63/63 [00:02<00:00, 30.74it/s]


(3978, 384)

In [ ]:
print(questions[0])
print(embeddings[0][:5])

What is the step by step guide to invest in share market in india?
[ 0.06814992 -0.03966417 -0.06096722  0.00746608 -0.05872768]


In [ ]:
client = chromadb.PersistentClient(path=str(DB_PATH))

try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

ids = [f"q_{idx}" for idx in range(len(questions_list))]
metadatas = [{"source": "quora", "row": int(idx)} for idx in range(len(questions_list))]

collection.upsert(
    ids=ids,
    documents=questions_list,
    embeddings=embeddings.tolist(),
    metadatas=metadatas,
)

collection.count()

3978

In [ ]:
def semantic_search(query: str, top_k: int = 5) -> pd.DataFrame:
    query_embedding = model.encode([query], normalize_embeddings=True)[0]
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k,
    )

    return pd.DataFrame(
        {
            "id": results["ids"][0],
            "question": results["documents"][0],
            "distance": results["distances"][0],
        }
    )

In [ ]:
semantic_search("How can I become better at speaking English?", top_k=5)

,id,question,distance
0,q_1163,What can I do to improve my English speaking?,0.079425
1,q_2631,How can I improve my English speaking .?,0.082187
2,q_3149,How can I improve my spoken English?,0.087453
3,q_643,How can I improve my spoken English ability?,0.090339
4,q_3958,What is the best way to improve my spoken Engl...,0.118296


In [ ]:
test_queries = [
    "How do I learn Python programming?",
    "What is the best way to lose weight?",
    "How can I improve my communication skills?",
    "How do I prepare for a job interview?",
]

for q in test_queries:
    print(f"\nQuery: {q}")
    display(semantic_search(q, top_k=3))
# smaller distance means more similar. 


Query: How do I learn Python programming?


,id,question,distance
0,q_1183,How do I learn Python systematically?,0.128583
1,q_2740,How should you start learning programming?,0.272625
2,q_3169,"Starting with no programming experience, how l...",0.326202



Query: What is the best way to lose weight?


,id,question,distance
0,q_3266,What are the best ways to lose weight?,0.020290
1,q_1358,What is the best method of losing weight?,0.086705
2,q_1900,What is the best and quick way to lose weight?,0.103039



Query: How can I improve my communication skills?


,id,question,distance
0,q_685,How can I improve my skills?,0.290656
1,q_2672,How do I improve my learning skills?,0.365334
2,q_2164,How can I speak with more clarity and confidence?,0.365811



Query: How do I prepare for a job interview?


,id,question,distance
0,q_487,How do I prepare for software interviews?,0.212700
1,q_2476,What are the best ways to prepare for software...,0.231988
2,q_2961,What are some tips on making it through the jo...,0.299310
